# TwelveLabs Pegasus 1.5 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Pegasus 1.5 reads a video and answers in text. It takes in what it sees, what is said and
what is written on screen in a single pass; the clip used here is silent, so the cells
below exercise the pictures and the on-screen text (section 4) but not speech. It has two
modes. In the default mode you ask a question about the video, pointing at images from the
prompt text if you have them (section 5), and get prose or JSON back. In
`time_based_metadata` mode you describe the moments you care about and it returns each one
with its start and end time, with no prompt at all (section 6).

| Model | `bedrock-runtime` ID | Inference profiles | Input | Max input | Max output | Notes |
|---|---|---|---|---|---|---|
| Pegasus 1.5 | `twelvelabs.pegasus-1-5-v1:0` | `us.` · `global.` | Video and a text prompt | 25 MB of base64 inline; up to 2 hours and under 2 GB from S3 | `maxOutputTokens` 512 to 98,304 | Video to text; time-coded segments |

IDs, profiles and the output range as of October 2026, measured in `us-east-1`; input
limits from the [Pegasus 1.5 model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page. Every request
goes through a profile, which the [Pegasus 1.5 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-twelvelabs-pegasus-v1-5.html) states:
"Pegasus v1.5 is available through cross-Region inference only. Requests must use the US
geographic inference profile (`us.twelvelabs.pegasus-1-5-v1:0`) or the global inference
profile (`global.twelvelabs.pegasus-1-5-v1:0`). In-Region on-demand inference is not
supported." The card ticks Geo inference from `us-east-1`, `us-east-2`, `us-west-1` and
`us-west-2`, and says the US profile "routes requests to us-east-1, us-east-2, and
us-west-2"; Global is ticked from 31 Regions (section 7). Unlike Pegasus 1.2, 1.5 has no
`eu.` profile, so from `eu-west-1` and `ap-northeast-1` it is reached through `global.`.
The card also gives a context window of 261,120 tokens, `Max output tokens: 98,304
tokens`, `Model lifecycle: Active` with a launch date of 8 October 2026 and "At least a
45-day EOL notice will be provided", and bills the model through AWS Marketplace:
"Charges appear on your AWS bill and in AWS Cost Explorer under the model provider (not
under Amazon Bedrock)." Card read 20 October 2026.

Pegasus is on `bedrock-runtime` only: the card ticks that endpoint and crosses
`bedrock-mantle`, and of the APIs it ticks Invoke and response streaming while crossing
Converse, Responses and Chat Completions (20 October 2026). So every call below is
**InvokeModel** or **InvokeModelWithResponseStream** with Pegasus's own request body.

**The video.** The notebook downloads [*Sparks*](https://opencontent.netflix.com/), a
short film from Netflix's Open Content programme, released under the
[Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/)
licence. It uses the smallest published encode Pegasus accepts, 432p AV1, 17.5 MB and 3
minutes 49 seconds, which is 23.3 MB once base64-encoded and so just inside the 25 MB
inline limit; the 270p and 342p encodes are refused for resolution (Things that differ).
That encode is a video-only stream with no audio track, so the film's soundtrack is not
part of it, and the download cell checks the track types and the frame size. Nothing below
therefore exercises the audio path; point the same requests at a video with a `soun` track
to see what Pegasus does with speech. The file is cached outside the repository and is not
part of it.
*Sparks* © Netflix, Inc., CC BY 4.0.

**Covered:** first call, streaming, structured output with a JSON Schema, reading text
on screen, a reference image in the prompt, time-coded segments with per-segment fields
and what they cost, the `global.` profile from outside the US, and sending the video from
S3.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with boto3.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import base64
import json
import os
import re
import urllib.request
from pathlib import Path

import boto3
from botocore.config import Config

REGION = "us-east-1"
PEGASUS = "twelvelabs.pegasus-1-5-v1:0"
PROFILE = f"us.{PEGASUS}"

# A video call takes 15 to 35 seconds here, so allow well past boto3's 60-second default.
runtime = boto3.client("bedrock-runtime", region_name=REGION,
                       config=Config(read_timeout=300, retries={"max_attempts": 3}))


### Download the video

The file comes from Netflix's public download bucket. The cell checks its size against
the published object, so a truncated download fails here rather than as a confusing
model error later. It also lists the file's track types: an MP4 names each track's kind
in a `hdlr` box, `vide` for video and `soun` for sound. It reads the frame size out of
the track header too and stops when either side is below 360 pixels, which is the floor
Pegasus accepts (Things that differ). It keeps the file under your home directory so a
re-run does not fetch it again.

In [2]:
VIDEO_URL = ("https://s3.amazonaws.com/download.opencontent.netflix.com/"
             "AV1/cmaf/spark-8b-59.94fps/spark_606kbps_432p.mp4")
VIDEO_BYTES = 17_497_474    # size of the published object, October 2026
INLINE_LIMIT = 25_000_000   # base64String limit from the Pegasus 1.5 model parameters page
MIN_SIDE = 360              # Pegasus refuses a smaller frame; see Things that differ
DURATION_SEC = 229.81       # length of this encode, 3 minutes 49.81 seconds

cache = Path.home() / ".cache" / "sample-per-model-bedrock" / "sparks_432p.mp4"
if not cache.exists() or cache.stat().st_size != VIDEO_BYTES:
    cache.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(VIDEO_URL, timeout=120) as resp:  # noqa: S310  # nosec B310 - fixed https URL
        cache.write_bytes(resp.read())
video = cache.read_bytes()
if len(video) != VIDEO_BYTES:
    raise RuntimeError(f"expected {VIDEO_BYTES} bytes, got {len(video)}; delete {cache} and re-run")
tracks = {m.group(1).decode() for m in re.finditer(rb"hdlr\x00{8}(....)", video)}
if tracks != {"vide"}:
    raise RuntimeError(f"expected a single video track, found {sorted(tracks)}")
# A track header ends with display width and height as 16.16 fixed point. Pegasus takes the
# frame size from the file, so read it here rather than meeting it as a 400 on the first call.
# Those four bytes occur inside media data too, so accept only a match whose box length is
# one a track header can have.
for match in re.finditer(rb"tkhd", video):
    start = match.start() - 4
    size = int.from_bytes(video[start:start + 4], "big")
    if 92 <= size <= 1024:
        tkhd = video[start:start + size]
        break
else:
    raise RuntimeError(f"no track header box in {cache}; cannot read the frame size")
width = int.from_bytes(tkhd[-8:-4], "big") >> 16
height = int.from_bytes(tkhd[-4:], "big") >> 16
if min(width, height) < MIN_SIDE:
    raise RuntimeError(f"{width}x{height}: Pegasus needs at least {MIN_SIDE} pixels on both sides")
VIDEO_B64 = base64.b64encode(video).decode()
print(f"{cache.name}: {len(video):,} bytes, {len(VIDEO_B64):,} as base64, "
      f"under the inline limit: {len(VIDEO_B64) <= INLINE_LIMIT}")
print(f"track types: {sorted(tracks)}; has audio: {'soun' in tracks}")
print(f"frame {width}x{height}; both sides at least {MIN_SIDE}: {min(width, height) >= MIN_SIDE}")

sparks_432p.mp4: 17,497,474 bytes, 23,329,968 as base64, under the inline limit: True
track types: ['vide']; has audio: False
frame 768x432; both sides at least 360: True


## 1. First call

The request body is Pegasus's own JSON: a prompt, the video under `mediaSource`, and
settings. `mediaSource` takes either `base64String`, as here, or an `s3Location` with
the object's `uri` and the `bucketOwner` account ID. The 25 MB inline limit applies to
the base64 string rather than to the file, and base64 adds a third, so inline tops out
near 18 MB of video: a 20 MB file returns `ValidationException: Input is too long for
requested model.` (October 2026). The clip here is 17.5 MB, 23.3 MB encoded, so it sits
just inside that. From S3 the model takes up to 2 hours of video under 2 GB, which is the
route for anything longer than a short clip (section 8). The answer is
in `message`, and `stopReason` is `stop` when it finished or `length` when it reached
`maxOutputTokens`.

The check looks for welding, which is what the worker in the film spends most of it
doing, rather than for exact words: wording changes between runs even at temperature 0.

In [3]:
def ask(prompt, model=PROFILE, client=runtime, mediaSource=None, **settings):
    # mediaSource defaults to the inline video; section 8 passes an s3Location instead.
    body = {"inputPrompt": prompt, "mediaSource": mediaSource or {"base64String": VIDEO_B64},
            "temperature": 0, **settings}
    response = client.invoke_model(modelId=model, body=json.dumps(body),
                                   contentType="application/json", accept="application/json")
    return json.loads(response["body"].read())


first = ask("Describe what happens in this video in three sentences.")
print(first["message"])
print(f"\nstopReason={first['stopReason']}  input_tokens={first['input_tokens']}  "
      f"output_tokens={first['output_tokens']}")
print("mentions welding:", "weld" in first["message"].lower())

The video begins with a sunrise over a forested landscape, transitioning to a construction worker in a cage elevator. The worker, equipped with safety gear, is shown welding steel beams at a construction site, with sparks flying. The video concludes with the worker taking a break and interacting with another worker, followed by a water truck spraying the site.

stopReason=stop  input_tokens=163  output_tokens=69
mentions welding: True


## 2. Streaming

`invoke_model_with_response_stream` sends the answer in chunks, each with a `message`
fragment. The last chunk carries `stopReason` and Bedrock's invocation metrics.

In [4]:
body = {"inputPrompt": "Count the people who appear in this video, then describe each in "
                       "one short sentence.",
        "mediaSource": {"base64String": VIDEO_B64}, "temperature": 0}
stream = runtime.invoke_model_with_response_stream(
    modelId=PROFILE, body=json.dumps(body),
    contentType="application/json", accept="application/json")
parts, last = [], {}
for event in stream["body"]:
    if "chunk" in event:
        last = json.loads(event["chunk"]["bytes"])
        parts.append(last.get("message", ""))
        print(parts[-1], end="", flush=True)
metrics = last.get("amazon-bedrock-invocationMetrics", {})
print(f"\n\nstopReason={last.get('stopReason')}  chunks={len(parts)}  metrics={metrics}")

There

 are two people in the

 video. The

 first is a construction worker wearing a

 hard hat and safety gear, seen

...

 the video. The second is a

 construction worker wearing a red shirt and

 hard hat, seen interacting with the

 first worker on a lift.



stopReason=stop  chunks=14  metrics={'inputTokenCount': 0, 'outputTokenCount': 55, 'invocationLatency': 27201, 'firstByteLatency': 10892}


## 3. Structured output with a JSON Schema

Set `responseFormat` to `{"type": "json_schema", "jsonSchema": {...}}`. Pegasus 1.5
requires the `type` field, which the Pegasus 1.2 request did not have. The
[model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page states: "When you include responseFormat, you must also set
responseFormat.type; omitting it returns a ValidationException." (20 October 2026)

`message` holds a JSON string that matches the schema.

The check asserts that every scene it calls welding lies between 60 and 180 seconds,
where sections 1 and 2 put it.

In [5]:
SCENES = {
    "type": "object",
    "properties": {"scenes": {"type": "array", "items": {
        "type": "object",
        "properties": {"start_sec": {"type": "number"}, "end_sec": {"type": "number"},
                       "description": {"type": "string"}},
        "required": ["start_sec", "end_sec", "description"]}}},
    "required": ["scenes"],
}
answer = ask("List the scenes of this video.",
             responseFormat={"type": "json_schema", "jsonSchema": SCENES})
scenes = json.loads(answer["message"])["scenes"]
for s in scenes:
    print(f"{s['start_sec']:6.0f} - {s['end_sec']:4.0f} s  {s['description']}")

welding = [s for s in scenes if re.search(r"\bweld(s|ed|ing)\b", s["description"].lower())]
print(f"\nwelding scenes: {len(welding)}; inside 60-180 s: "
      f"{bool(welding) and all(60 <= s['start_sec'] and s['end_sec'] <= 180 for s in welding)}")

     0 -    8 s  Sunrise over a forested landscape.
     8 -   24 s  A construction worker in a cage elevator preparing for his workday.
    24 -   43 s  The worker puts on his safety harness and gear.
    43 -   81 s  The worker walks through a construction site, ascends stairs, and prepares to weld.
    81 -  131 s  The worker welds a steel beam with sparks flying.
   131 -  149 s  The worker takes a break and drinks water.
   149 -  163 s  The worker talks with another worker on a lift.
   163 -  168 s  A worker sweeps the construction site.
   168 -  195 s  The worker walks towards his truck and drives away.
   195 -  230 s  The video ends with the credits.

welding scenes: 1; inside 60-180 s: True


## 4. Text on screen

Pegasus reads writing in the frame as well as the action. The film ends on credits on a
black screen, so the cell asks for the title shown there and checks it against the one
the credits carry, `SPARKS`.

In [6]:
title = ask("What is the title of the film, as written on screen in the closing credits? "
            "Answer with the title only.")
print(repr(title["message"]), " stopReason:", title["stopReason"])
print("reads the on-screen title:", title["message"].strip().strip(".").upper() == "SPARKS")

'SPARKS'  stopReason: stop
reads the on-screen title: True


## 5. A reference image in the prompt

`prompt_v2` replaces `inputPrompt` and lets the prompt text point at images. Each entry in
`media_sources` carries a `name`, a `media_type` of `image` and a `base64String`, and
`input_text` refers to an entry as `<@name>`. The images supplement the video, which is
still required: without `mediaSource` the request returns 400 `no video input provided —
send mediaSource (s3Location or base64String) or a top-level s3Location` (23 October 2026).
This is the shape the [model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page documents, and it states: "To refer to
an image, include a placeholder of the form `<@name>`, where name matches the name of an
entry in media_sources. A placeholder with no matching entry returns a
ValidationException."

The image here is the invoice that ships with this repository. The prompt asks for two
fields only the image carries and one fact only the video carries, so each part of the
answer can be checked against a known value. A placeholder written another way is accepted
rather than refused, and the fields come back unread (Things that differ), which is why the
cell checks the answer rather than printing it alone.

In [7]:
import sys

sys.path.insert(0, "../_shared")
from bedrock import INVOICE_FIELDS, invoice_png

PROMPT_V2 = {
    "input_text": ("Read the invoice number and the vendor name written in <@invoice>, then "
                   "say whether that document appears anywhere in this video. "
                   "Answer as NUMBER | VENDOR | yes or no."),
    # `name` is what <@invoice> resolves to. A placeholder matching no entry returns 400;
    # one written another way is accepted and the image goes unread.
    "media_sources": [{"name": "invoice", "media_type": "image",
                       "base64String": base64.b64encode(invoice_png()).decode()}],
}
body = {"prompt_v2": PROMPT_V2, "mediaSource": {"base64String": VIDEO_B64}, "temperature": 0}
response = runtime.invoke_model(modelId=PROFILE, body=json.dumps(body),
                                contentType="application/json", accept="application/json")
ref = json.loads(response["body"].read())
parts = [p.strip() for p in ref["message"].split("|")]
if len(parts) != 3:
    raise RuntimeError(f"expected 'NUMBER | VENDOR | yes or no', got {ref['message']!r}")
print(ref["message"], " stopReason:", ref["stopReason"])
print("invoice number from the image:", parts[0] == INVOICE_FIELDS["invoice_id"])
print("vendor from the image:", parts[1] == INVOICE_FIELDS["vendor"])
print("the film does not show that document:", parts[2].lower().startswith("no"))

INV-1042 | Acme Pty Ltd | no  stopReason: stop
invoice number from the image: True
vendor from the image: True
the film does not show that document: True


## 6. Time-coded segments

Pegasus 1.2 has no equivalent of this mode. Set `analysis_mode` to `time_based_metadata`
and describe each kind of moment in `segment_definitions`: an `id`, a `description` of
what to find, and a list of `fields`, each with a `name`, `type` and `description`, that
Pegasus fills in for every segment it finds. Each returned segment carries `start_sec`,
`end_sec`, one key per field name and, when you send more than one definition,
`_definition_id`, the `id` it matched; with a single definition the service omits it, so
read it with `.get`. This is the
shape the [Pegasus 1.5 model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page documents. The service also accepts
`analysisMode` beside `responseFormat: {"type": "segment_definitions",
"segment_definitions": [...]}` and returns the same segments (October 2026).

This mode takes no prompt: adding `inputPrompt` returns a 400. It also reads far more of
the video: the input tokens below sit beside section 3's for the same file, and the gap is
roughly two orders of magnitude.

A segment says when something happens, not whether it happens. Pegasus returns a segment
for every definition, including one with no basis in the footage, so the cell checks the
times rather than treating a returned definition as a match.

In [8]:
DEFINITIONS = [
    {"id": "welding", "description": "The worker is welding a steel beam",
     "fields": [{"name": "sparks_visible", "type": "string",
                 "description": "Are sparks visible in this segment"}]},
    {"id": "break", "description": "The worker drinks water or talks with a colleague",
     "fields": [{"name": "who_else", "type": "string",
                 "description": "Who else appears in this segment"}]},
]
body = {"mediaSource": {"base64String": VIDEO_B64}, "temperature": 0,
        "analysis_mode": "time_based_metadata", "segment_definitions": DEFINITIONS}
response = runtime.invoke_model(modelId=PROFILE, body=json.dumps(body),
                                contentType="application/json", accept="application/json")
timed = json.loads(response["body"].read())
FIELDS = {d["id"]: [f["name"] for f in d["fields"]] for d in DEFINITIONS}
for seg in timed["segments"]:
    did = seg.get("_definition_id", DEFINITIONS[0]["id"])   # absent with one definition
    values = ", ".join(f"{name}={seg.get(name)!r}" for name in FIELDS[did])
    print(f"{did:8} {seg['start_sec']:6.0f} - {seg['end_sec']:4.0f} s  {values}")
print("definition_errors:", timed["definition_errors"], " stopReason:", timed["stopReason"])

print("segments inside the clip: "
      f"{all(0 <= s['start_sec'] < s['end_sec'] <= DURATION_SEC for s in timed['segments'])}")
print("every segment carries its fields: "
      f"{all(all(n in s for n in FIELDS[s.get('_definition_id', DEFINITIONS[0]['id'])]) for s in timed['segments'])}")
overlap = any(seg.get("_definition_id") == "welding" and seg["start_sec"] < s["end_sec"]
              and s["start_sec"] < seg["end_sec"] for seg in timed["segments"] for s in welding)
print(f"welding segment overlaps section 3's welding scene: {overlap}")
print(f"input tokens: {timed['input_tokens']:,} here, {answer['input_tokens']:,} for section 3 "
      f"({timed['input_tokens'] / answer['input_tokens']:.0f}x)")

welding      83 -  130 s  sparks_visible='Yes'
break       140 -  148 s  who_else='none'
break       149 -  162 s  who_else='colleague in red shirt'
definition_errors: []  stopReason: stop
segments inside the clip: True
every segment carries its fields: True
welding segment overlaps section 3's welding scene: True
input tokens: 79,773 here, 529 for section 3 (151x)


## 7. The `global.` profile from outside the US

Pegasus 1.5 has a `us.` profile and a `global.` one. From a Region outside the US, such as
`eu-west-1` or `ap-northeast-1`, only `global.` exists. The model card ticks Global
inference from 31 Regions as of 20 October 2026, and `me-central-1`, `me-south-1` and
`ap-east-1` are not among them, so from those three neither profile is available. Global
routing can serve the request from a Region other than the one you call, so check that
against your data rules. The base ID refuses on-demand calls in every Region, which is
what the card means by "In-Region on-demand inference is not supported".

In [9]:
eu = boto3.client("bedrock-runtime", region_name="eu-west-1",
                  config=Config(read_timeout=300, retries={"max_attempts": 3}))
reply = ask("In one sentence, what is the worker doing for most of the video?",
            model=f"global.{PEGASUS}", client=eu)
print(reply["message"])
print("mentions welding:", "weld" in reply["message"].lower())

try:
    ask("One sentence.", model=PEGASUS)
except runtime.exceptions.ValidationException as exc:
    print("\nbase ID:", str(exc).split(": ", 1)[1])

The worker is welding steel beams at a construction site.
mentions welding: True



base ID: Invocation of model ID twelvelabs.pegasus-1-5-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.


## 8. Video from S3

`mediaSource` takes either the base64 string the cells above send or an `s3Location`,
which is the route to the model's full input range of up to 2 hours of video under
2 GB. It needs two fields together, the object's `uri` and `bucketOwner`, the account ID
of the bucket's owner. Sending one without the other returns 400
`ValidationException: Invalid JSON: s3Location requires both uri and bucketOwner`
(October 2026). The bucket has to be in the Region you send the request to, which the
[model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page states: "The bucket must be in the same AWS Region that you
send the request to. This applies to the geo and global inference profiles as well: the
profile determines where the request can be served, but it does not widen where the
bucket can live. A bucket in any other Region fails with a ValidationException."

The first cell sends the request when `PEGASUS_VIDEO_URI` names an object Bedrock can
read, and prints a line saying so otherwise, which is what the committed output shows:
the account these cells ran on has no S3 permissions, so the request shape is committed
unexecuted, and the Region and credentials you run it with decide whether the object can
be read. Set `BUCKET_OWNER` as well when the bucket belongs to another account.

The second cell is the part a reader is most likely to meet, and the reason to check the
account ID before the key: a `bucketOwner` that does not own the bucket, a typo in the
URI and an object that is not there all return the same `Provided S3Location not found`,
with nothing in the message to tell them apart (Things that differ).

In [10]:
# Point PEGASUS_VIDEO_URI at a video Bedrock can read to send this request. bucketOwner
# is required beside uri, and this account owns the bucket in the common case, so take
# it from STS unless BUCKET_OWNER says otherwise. The ID is not printed: an account ID
# in a committed notebook output is an account ID published to this repository.
uri = os.environ.get("PEGASUS_VIDEO_URI")
owner = os.environ.get("BUCKET_OWNER")
if uri and not owner:
    sts = boto3.client("sts", region_name=REGION, config=Config(read_timeout=10))
    owner = sts.get_caller_identity()["Account"]

if uri:
    s3_answer = ask("In one sentence, what is the main subject of this video doing?",
                    mediaSource={"s3Location": {"uri": uri, "bucketOwner": owner}})
    print(s3_answer["message"])
    print(f"stopReason={s3_answer['stopReason']}  input_tokens={s3_answer['input_tokens']}")
else:
    print("PEGASUS_VIDEO_URI is not set, so no S3 request was sent")

PEGASUS_VIDEO_URI is not set, so no S3 request was sent


In [11]:
# 123456789012 is this repository's placeholder account ID, and the bucket name uses the
# documented example form, so neither names anything real.
MISSING_URI = "s3://amzn-s3-demo-bucket/no-such-video.mp4"
cases = {
    "uri without bucketOwner": {"s3Location": {"uri": MISSING_URI}},
    "wrong owner or no object": {"s3Location": {"uri": MISSING_URI,
                                                "bucketOwner": "123456789012"}},
}
for label, media in cases.items():
    try:
        ask("One sentence.", mediaSource=media)
    except runtime.exceptions.ValidationException as exc:
        print(f"{label:25} {str(exc).split(': ', 1)[1]}")
    else:
        # Both cases are refusals today; a success here means the surface changed.
        raise RuntimeError(f"{label}: expected a ValidationException, got an answer")

uri without bucketOwner   Invalid JSON: s3Location requires both uri and bucketOwner 


wrong owner or no object  Provided S3Location not found


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Base model ID | 400 `ValidationException`: `Invocation of model ID twelvelabs.pegasus-1-5-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.` (October 2026) | Call `us.twelvelabs.pegasus-1-5-v1:0` or `global.twelvelabs.pegasus-1-5-v1:0`. |
| No `eu.` profile | Pegasus 1.2 has `eu.twelvelabs.pegasus-1-2-v1:0`; 1.5 has only `us.` and `global.` (October 2026) | From Europe or Asia Pacific use `global.`, and check whether Global routing, which can serve the request from a Region other than the one you call, is acceptable for your data. The card ticks Global inference from 31 Regions (20 October 2026), without `me-central-1`, `me-south-1` or `ap-east-1`. |
| Converse | 400 `ValidationException`: `This action doesn't support the model that you provided. Try again with a supported text or chat model.` | Use `InvokeModel` or `InvokeModelWithResponseStream` with Pegasus's own request body. |
| Response field names | The [model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page documents five response fields, `message`, `stopReason`, `segments`, `definition_errors` and `finish_reason`. The calls below also return `input_tokens` and `output_tokens`, which are on neither that page nor the model card (20 October 2026). Each segment carries `start_sec`, `end_sec` and one key per declared field, plus `_definition_id` when more than one definition was sent; a definition without `fields` returns its `description` instead. The page states: "The segment identifier is named `_definition_id`, not `id`. The `id` field appears only on the request side, in each `segment_definitions` entry." | Read `stopReason`, which both modes return, read segment values by the field names you declared, and read `_definition_id` with `.get`. Treat the two token counts as undocumented. |
| `responseFormat` without `type` | 400: `The responseFormat.type parameter is invalid. Field required`, which the [model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page states: "When you include responseFormat, you must also set responseFormat.type; omitting it returns a ValidationException." An unknown type returns `Input should be 'json_schema' or 'segment_definitions'` | Send `{"type": "json_schema", "jsonSchema": {...}}`, the 1.2 shape plus `type`. |
| Output budget | `maxOutputTokens` 20 returns 400 `max_tokens must be at least 512`; 100000 returns `max_tokens must be at most 98304`. 512, 4096 and 8192 are accepted (October 2026). An answer cut off by the budget ends with `stopReason` `length` | Keep `maxOutputTokens` between 512 and 98,304, and treat `length` as an incomplete answer. The card gives the same ceiling, `Max output tokens: 98,304 tokens`. |
| A placeholder written another way | `{{invoice}}` in place of `<@invoice>` returns 200, and the answer comes back with the image's fields unread: `N/A` for both the invoice number and the vendor, where the documented form read `INV-1042` and `Acme Pty Ltd` on each of three runs (23 October 2026, `us-east-1`). A placeholder that names no entry does return 400: `The prompt_v2.input_text parameter is invalid. placeholder '<@missing>' not found in media_sources` | Write the placeholder as `<@name>`, and check the answer against something only the image carries, because the wrong form is not refused. |
| `inputPrompt` with `prompt_v2` | 400 `ValidationException`: `The prompt parameter is invalid. prompt and prompt_v2 are mutually exclusive`, which names `prompt` rather than the `inputPrompt` field you sent (23 October 2026) | Send one or the other, and in `time_based_metadata` mode neither (the next row). |
| Segments with a prompt | `analysis_mode=time_based_metadata does not accept prompt/prompt_v2 — use response_format.segment_definitions or custom_fields` | Leave `inputPrompt` out in that mode; put what you are looking for in each definition's `description`. |
| Segments without the mode | `response_format.type must be json_schema for analysis_mode=general` | Set `"analysis_mode": "time_based_metadata"` beside `segment_definitions`. The accepted modes are `general` and `time_based_metadata`; `analysisMode` is accepted too. |
| Segment definition fields | A definition-level `name` returns `Extra inputs are not permitted`, and a definition without `id` returns `Field required`; an empty list returns `segment_definitions is required (non-empty)` | Give each definition an `id`, a `description` and, for structured values per segment, `fields` of `name`, `type` and `description`. |
| A definition that never occurs | Two definitions with no basis in the footage, a scuba diver and night snowfall, each came back as a segment with `definition_errors` still `[]`. One of the two filled its field with `No segments matched the specified criterion.` while the segment itself still spanned the whole clip (19 October 2026) | Segments tell you when something happens, not whether it happens. Do not use this mode to detect absence; verify a match against the segment's field values or a second call. |
| Cost of timestamps | On this 3:49 film the segment request read about 79,770 input tokens, between 79,765 and 79,773 on every run measured for this row, including the one committed in section 6, against about 460 to 610 for the JSON Schema question in section 3, which moves from run to run: 529 and 607, each of them twice, across four runs here, and 464 and 583 in runs on 22 October 2026. That is about 130 to 170 times as many, on the 432p encode the notebook sends (23 October 2026, `us-east-1`). The same two requests returned 119,597 and 583 on 20 October 2026, so both counts move with the service and the video side fell by a third | Use segments when you need the times, and plain questions for everything else. Meter the call rather than taking a figure from here. `start_time` and `end_time` ([model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html)) cut the cost of this mode when part of the clip will do: a 60 to 90 second window read 12,717 to 12,725 input tokens across four runs, against about 79,770 for the whole clip. The saving is in this mode. The same window moved section 3's question from 529 and 607 to between 363 and 420, and section 1's prompt from 163 and 170 to 144 and 177, which is inside its own run-to-run spread. A window does not confine what comes back, although the page above states: "Applies to both analysis modes; in `time_based_metadata` mode only segments inside the clip are returned". In segment mode 2 of 4 windowed runs returned a segment outside the window, at 130 to 140 s and 131 to 140 s for a 60 to 90 s request, so filter on `start_sec` and `end_sec` yourself. In `general` mode the base the timestamps are counted from is not stable: across three sessions of five runs each in `us-east-1` on 23 October 2026, the same windowed request numbered its scenes from the window start, 0 to 29 s for a 60 to 90 s request, in 7 of the 15 runs, and from the start of the video in the other 8, so read the times against the window you asked for rather than expecting either base. |
| Service tiers | `"service_tier": "flex"`, `"priority"` and `"default"` all return 400 `The service_tier parameter is invalid. Extra inputs are not permitted`. The card ticks Standard alone, crossing Priority, Flex, Reserved and Batch, and says of Standard: "set `"service_tier": "default"` or omit the field" (card read 20 October 2026), so the service refuses even the spelling the card gives (October 2026) | Leave `service_tier` out of Pegasus requests. |
| Streaming token count | The last chunk's `amazon-bedrock-invocationMetrics` reports `inputTokenCount` 0, while the same prompt without streaming reports `input_tokens` 163 (19 October 2026) | Do not meter input from the streaming metrics; use the non-streaming fields or CloudWatch. |
| Temperature 0 | The same question returned different wording on each run (October 2026) | Check facts in the answer, such as a keyword or a time range, rather than its exact text. |
| Video resolution floor | 400 `ValidationException`: `The resolution of the video is too low. Please upload a video with resolution between 360p(360x360) and 2160p(5184x2160). Current resolution is 480x270.` The 270p (480x270) and 342p (608x342) *Sparks* encodes are both refused, the 432p (768x432) encode this notebook sends is accepted (19 October 2026, `us-east-1`) | Send a frame at least 360 pixels on both sides. The floor is not on the [model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html) page, so the message is the only statement of it, which is why the download cell reads the frame size and stops before the call. |
| Inline video size | The 25 MB limit on `base64String` ([model parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus-1-5.html)) is on the encoded string, so about 18 MB of video. A 20 MB file returned `ValidationException: Input is too long for requested model.` (October 2026) | Send anything above about 18 MB from S3 with `mediaSource.s3Location` (`uri` and `bucketOwner`): up to 2 hours of video, under 2 GB (section 8). |
| `s3Location` fields | Either field without the other returns 400 `ValidationException: Invalid JSON: s3Location requires both uri and bucketOwner`. A `bucketOwner` that does not own the bucket, a URI typo and a missing object all return the same 400 `ValidationException: Provided S3Location not found` (October 2026) | Send `uri` and `bucketOwner` together, and on `Provided S3Location not found` check the account ID as well as the key, because one message covers a wrong owner, a typo and a missing object. Keep the bucket in the Region you call, whichever profile you use (section 8). |

## Next

- Images rather than video: [`12-writer-palmyra/01`](../12-writer-palmyra/01-palmyra-vision.ipynb),
  [`13-amazon-nova/01`](../13-amazon-nova/01-nova-ladder-and-vision.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)